# TP1 -- Graphiques Quantile-Quantile, Test de Normalité et Test du χ²

**Statistique Mathématique 3 -- L3 MIASHS**

---

## Introduction

Dans ce TP, nous allons utiliser les outils de la **normalité** et de **l'indépendance** dans deux cadres :

- Sur **données ordonnées** (regroupées par modalités) : nous reconstruirons tous les outils à la main.
- Sur **données brutes** : nous utiliserons largement les fonctions de R.

**Objectifs pédagogiques :**
- Comprendre et tracer un QQ-plot
- Réaliser un test du χ² d'adéquation (normalité)
- Réaliser un test du χ² d'indépendance
- Utiliser les fonctions R pour tester la normalité (Shapiro-Wilk, `qqnorm`)

---
## 1. Le QQ-Plot (graphique quantile-quantile)

### 1.1 Rappels théoriques

Le **QQ-plot** est un outil graphique qui permet de visualiser si une série de données suit une distribution théorique (gaussienne, Poisson, etc.).

**Principe :** On trace les fractiles observés ($q_i$) en ordonnées contre les fractiles théoriques ($q_i^*$) en abscisses.

**Interprétation :** Plus les points sont alignés sur la **diagonale**, plus l'adéquation est forte.

### 1.2 Application : tensions de pile

Xantane a relevé 500 fois les tensions de sa pile. Les données sont regroupées :

| $x_i$ | 92 | 93 | 94 | 95 | 96 | 97 | 98 | 99 | 100 | 101 | 102 | 103 | 104 | 105 | 106 | 107 | 108 | 109 |
|--------|----|----|----|----|----|----|----|----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| $n_i$  |  1 |  4 |  9 | 28 | 23 | 37 | 51 | 63 |  65 |  60 |  48 |  38 |  32 |  23 |  12 |   3 |   1 |   2 |

In [ ]:
# Saisie des données
qi = 92:109
ni = c(1, 4, 9, 28, 23, 37, 51, 63, 65, 60, 48, 38, 32, 23, 12, 3, 1, 2)

In [ ]:
# Calcul de la moyenne pondérée : m = Σ(qi*ni) / Σ(ni)
m = sum(qi * ni) / sum(ni)
cat("Moyenne pondérée m =", m)

In [ ]:
# Calcul de l'écart-type pondéré
sigma = sqrt(sum(ni * (qi - m)^2) / (sum(ni) - 1))
cat("Écart-type sigma =", round(sigma, 2))

In [ ]:
# Fréquences cumulées et quantiles théoriques
Fi = cumsum(ni) / sum(ni)
qietoile = qnorm(Fi, m, sigma)

In [ ]:
# Tracé du QQ-plot
plot(qi, qietoile,
     main = "QQ-Plot : Tensions de pile",
     xlab = "Quantiles observés (qi)",
     ylab = "Quantiles théoriques (qi*)",
     pch = 19, col = "blue")
lines(c(90, 110), c(90, 110), col = "red", lwd = 2)
legend("topleft", legend = "Diagonale y = x", col = "red", lwd = 2)

**Conclusion :** Les points sont bien alignés sur la diagonale → bonne adéquation avec $\mathcal{N}(m, \sigma)$.

---
## 2. Test du χ² pour la normalité

Puisque nous n'avons pas les données brutes, on ne peut pas utiliser Shapiro-Wilk. On utilise le **test du χ²**.

### 2.1 Fusion des effectifs faibles
Pour que le test soit valide, les effectifs théoriques doivent être > 5.

In [ ]:
# Fusion des 2 premiers et 3 derniers effectifs
ni2 = c(ni[1] + ni[2], ni[3:15], ni[16] + ni[17] + ni[18])
cat("Effectifs regroupés :", ni2)

In [ ]:
# Probabilités théoriques avec correction de continuité
k = c(91, 93:106) + 0.5
K = c(93:106.5, 109) + 0.5
pi_theo = pnorm(K, 100, 3.1) - pnorm(k, 100, 3.1)
ti = 500 * pi_theo

In [ ]:
# Statistique du χ²
dchi = sum((ni2 - ti)^2 / ti)
cat("Statistique du chi-deux :", round(dchi, 4))

# Avec la fonction R (attention aux d.d.l.)
chisq.test(ni2, p = pi_theo)

In [ ]:
# Degrés de liberté corrects : 15 - 1 - 2 = 12
cat("Quantile théorique χ²(0.95, 12) =", qchisq(0.95, 12))
cat("\nStatistique observée =", round(dchi, 4))
cat("\np-value correcte =", 1 - pchisq(dchi, 12))

**Conclusion :** p-value > 0.05 → on **ne rejette pas** l'hypothèse de normalité au seuil 5%.

---
## 3. Test du χ² d'indépendance

### 3.1 Importation des données du Titanic

Le fichier `titanic.csv` contient le registre des passagers. Objectif : étudier la relation entre **survie** et **classe**.

In [ ]:
titanic = read.csv("titanic.csv")
head(titanic)

In [ ]:
survived = titanic$Survived
class = titanic$Pclass

### 3.2 Test d'indépendance : Survie vs Classe

In [ ]:
# Tableau de contingence
contingence1 = table(survived, class)
contingence1

In [ ]:
# Test du χ² d'indépendance
chisq.test(contingence1)

**Interprétation :** p-value ≈ 0 → on rejette H₀ → la classe et la survie sont **dépendantes**.

### 3.3 Autres tests d'indépendance

In [ ]:
# Survie vs Sexe
contingence_sex = table(survived, titanic$Sex)
contingence_sex
chisq.test(contingence_sex)

In [ ]:
# Classe vs Port d'embarquement (on enlève la colonne vide)
contingence2 = table(class, titanic$Embarked)
contingence2 = contingence2[, -1]
contingence2
chisq.test(contingence2)

In [ ]:
# Survie vs Port d'embarquement
contingence3 = table(survived, titanic$Embarked)
contingence3 = contingence3[, -1]
chisq.test(contingence3)

---
## 4. Normalité et fonctions de R

### 4.1 Acquisition de la variable `age`

In [ ]:
age = titanic$Age
head(age)
# Supprimer les valeurs manquantes
age = age[!is.na(age)]
class2 = class[!is.na(titanic$Age)]
cat("Nombre d'âges valides :", length(age))

### 4.2 Normalité de `age`

In [ ]:
# Densité estimée
plot(density(age), main = "Densité de la variable Age",
     xlab = "Âge", col = "blue", lwd = 2)

In [ ]:
# QQ-plot
qqnorm(age, main = "QQ-plot de Age")
qqline(age, col = "red")

In [ ]:
# Test de Shapiro-Wilk
shapiro.test(age)

**Conclusion :** La variable `age` **n'est pas normale** (densité non gaussienne, QQ-plot déviant, p-value Shapiro < 0.05).

### 4.3 Normalité de `age` par classe

In [ ]:
# Test de Shapiro-Wilk par classe
tapply(age, class2, shapiro.test)

**Conclusion :** Seul en **première classe**, l'âge peut être considéré comme normal.

---
## Résumé

| Outil | Utilisation | Fonction R |
|-------|-------------|------------|
| QQ-plot | Visualiser l'adéquation à une loi | `qqnorm()`, `plot()` |
| Test χ² d'adéquation | Tester si les données suivent une loi | `chisq.test()` |
| Test χ² d'indépendance | Tester l'indépendance de 2 variables | `chisq.test(table())` |
| Test de Shapiro-Wilk | Tester la normalité (données brutes) | `shapiro.test()` |
| `tapply` | Appliquer une fonction par sous-groupe | `tapply(var, groupe, fun)` |